# TP9 — Validation GroupKFold par machine

Dans TP8b/TP8c, le split temporel place les **mêmes machines** en train et val.  
Le modèle peut apprendre les signatures spécifiques de chaque machine → biais structurel.

GroupKFold simule le vrai scénario de production : **déployer sur une machine jamais vue**.

| Modèle | Validation | PR-AUC val | PR-AUC test | Overfitting Δ |
|--------|-----------|------------|-------------|---------------|
| B8 (TP8b) | Split temporel | 0.8497 | 0.8232 | 0.1503 |
| **B9-GKF** | GroupKFold | ? | ? | ? |

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

## 2. Données

In [ ]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username="indusense_user",
    password="ThEP@ssW0rd",
    host="localhost", port=5432, database="indusense_db",
)
engine = create_engine(url)
df = pd.read_sql(
    "SELECT * FROM gold_machine_hourly_feature ORDER BY machine_id, window_start",
    engine
)

TARGET = "label_failure_next_24h"
LEAKAGE_COLS = [
    "machine_id", "ingestion_batch_id", "window_start", "window_end", "split_set",
    "label_failure_next_6h", "label_failure_next_12h", "label_failure_next_48h",
    TARGET, "feature_row_id",
]
FEATURE_COLS = [c for c in df.columns if c not in LEAKAGE_COLS]

# Train + val pour le CV, test reste holdout final
trainval_df = df[df["split_set"].isin(["train", "validation"])].copy()
test_df     = df[df["split_set"] == "test"].copy()

X_tv   = trainval_df[FEATURE_COLS]
y_tv   = trainval_df[TARGET]
groups = trainval_df["machine_id"]

X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

machines = groups.unique()
N_MACHINES = len(machines)

print(f"Train+val : {len(X_tv):,} lignes | Test : {len(X_test):,} lignes")
print(f"Machines  : {N_MACHINES} → {sorted(machines)}")
print(f"Features  : {len(FEATURE_COLS)}")

# Positifs par machine
pos_per_machine = trainval_df.groupby("machine_id")[TARGET].sum().sort_values()
print("\nPositifs par machine :")
print(pos_per_machine.to_string())

## 3. GroupKFold CV — B9-GKF

Chaque fold exclut une machine entière du train et l'utilise comme val.  
On réutilise les hyperparamètres B8 (meilleurs params Optuna TP8b).  
`scale_pos_weight` est recalculé à chaque fold sur les données train du fold.

In [ ]:
# Hyperparamètres B8
B8_PARAMS = {
    "max_depth":         8,
    "learning_rate":     0.19367782999811112,
    "n_estimators":      382,
    "subsample":         0.9870283480802571,
    "colsample_bytree":  0.8035421212629575,
    "min_child_weight":  6,
    "reg_alpha":         0.0014393622771964793,
    "reg_lambda":        0.00014158997638181163,
    "random_state":      RANDOM_STATE,
    "verbosity":         0,
}

gkf = GroupKFold(n_splits=N_MACHINES)
fold_results = []

for fold_i, (tr_idx, vl_idx) in enumerate(gkf.split(X_tv, y_tv, groups)):
    machine_val = groups.iloc[vl_idx].unique()[0]

    X_tr, y_tr = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
    X_vl, y_vl = X_tv.iloc[vl_idx], y_tv.iloc[vl_idx]

    n_pos = int(y_vl.sum())

    # scale_pos_weight recalculé sur le fold train
    spw = round((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 2)

    params = {**B8_PARAMS, "scale_pos_weight": spw}
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   XGBClassifier(**params)),
    ])
    pipe.fit(X_tr, y_tr)

    y_prob = pipe.predict_proba(X_vl)[:, 1]

    if n_pos < 2:
        pr_auc = float("nan")
    else:
        pr_auc = average_precision_score(y_vl, y_prob)

    fold_results.append({
        "fold":      fold_i + 1,
        "machine":   machine_val,
        "n_obs":     len(y_vl),
        "n_pos":     n_pos,
        "pr_auc":    round(pr_auc, 4),
    })
    print(f"Fold {fold_i+1:2d} | {machine_val:<10} | n_pos={n_pos:4d} | PR-AUC={pr_auc:.4f}")

results_df = pd.DataFrame(fold_results)
valid = results_df.dropna(subset=["pr_auc"])
print(f"\nPR-AUC CV moyen  : {valid['pr_auc'].mean():.4f}")
print(f"PR-AUC CV std    : {valid['pr_auc'].std():.4f}")
print(f"PR-AUC CV min    : {valid['pr_auc'].min():.4f}")
print(f"PR-AUC CV max    : {valid['pr_auc'].max():.4f}")

## 4. Distribution des scores par machine

In [ ]:
valid_sorted = valid.sort_values("pr_auc")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#E05A5A" if v < 0.70 else "#E8B84B" if v < 0.82 else "#5ECFA8"
          for v in valid_sorted["pr_auc"]]
bars = ax.barh(valid_sorted["machine"], valid_sorted["pr_auc"],
               color=colors, height=0.6, edgecolor="none")

# Ligne de référence B8 val
ax.axvline(0.8497, color="#9B7FD4", linestyle="--", linewidth=1.5,
           label="B8 split temporel (val=0.8497)")

for bar, v in zip(bars, valid_sorted["pr_auc"]):
    ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{v:.3f}", va="center", fontsize=8)

ax.set_xlabel("PR-AUC")
ax.set_title("B9-GKF — PR-AUC par machine (fold val)")
ax.set_xlim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.savefig("b9_gkf_per_machine.png", dpi=150)
plt.show()

## 5. Modèle final — réentraîné sur tout train+val, évalué sur test

In [ ]:
SPW_FULL = round((y_tv == 0).sum() / (y_tv == 1).sum(), 2)
params_final = {**B8_PARAMS, "scale_pos_weight": SPW_FULL}

pipe_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   XGBClassifier(**params_final)),
])
pipe_final.fit(X_tv, y_tv)

y_prob_train = pipe_final.predict_proba(X_tv)[:, 1]
y_prob_test  = pipe_final.predict_proba(X_test)[:, 1]

pr_train = average_precision_score(y_tv, y_prob_train)
pr_test  = average_precision_score(y_test, y_prob_test)
roc_test = roc_auc_score(y_test, y_prob_test)
f1_test  = f1_score(y_test, pipe_final.predict(X_test), zero_division=0)
delta_cv_test = round(valid["pr_auc"].mean() - pr_test, 4)

print("=== Modèle final B9-GKF ===")
print(f"  PR-AUC train     : {pr_train:.4f}")
print(f"  PR-AUC CV moyen  : {valid['pr_auc'].mean():.4f} ± {valid['pr_auc'].std():.4f}")
print(f"  PR-AUC test      : {pr_test:.4f}")
print(f"  ROC-AUC test     : {roc_test:.4f}")
print(f"  F1 test          : {f1_test:.4f}")
print(f"  Δ CV-test        : {delta_cv_test:.4f}")

print("\n=== Comparaison globale ===")
print(f"  {'Modèle':<14} {'Val strategy':<20} {'PR-AUC val/CV':>14} {'PR-AUC test':>13}")
print("  " + "-"*65)
print(f"  {'B8 (TP8b)':<14} {'Split temporel':<20} {'0.8497':>14} {'0.8232':>13}")
print(f"  {'B9-GKF':<14} {'GroupKFold':<20} {valid['pr_auc'].mean():>14.4f} {pr_test:>13.4f}")

## 6. Analyse MACH-07 — Pourquoi PR-AUC = 0.329 ?

Trois hypothèses à tester :
- **H1** : trop peu de positifs dans val → score statistiquement non fiable
- **H2** : features de MACH-07 sont atypiques → outlier dans l'espace feature
- **H3** : le modèle prédit dans le mauvais sens → les pannes ressemblent à des non-pannes des autres machines

In [ ]:
# H1 — Statistiques de base MACH-07
mach07 = df[df["machine_id"] == "MACH-07"].copy()

stats = {}
for split in ["train", "validation", "test"]:
    sub = mach07[mach07["split_set"] == split]
    n_obs = len(sub)
    n_pos = int(sub[TARGET].sum())
    taux  = round(n_pos / n_obs * 100, 1) if n_obs > 0 else 0
    stats[split] = {"n_obs": n_obs, "n_pos": n_pos, "taux_%": taux}

fleet_taux = round(trainval_df[TARGET].mean() * 100, 1)

print("=== MACH-07 — Distribution des classes ===")
print(f"  {'Split':<12} {'n_obs':>8} {'n_pos':>8} {'taux_%':>10}")
print("  " + "-"*42)
for split, s in stats.items():
    print(f"  {split:<12} {s['n_obs']:>8,} {s['n_pos']:>8} {s['taux_%']:>9.1f}%")
print(f"\n  Taux de panne flotte (train+val) : {fleet_taux}%")

# Verdict H1
n_pos_val = stats["validation"]["n_pos"]
if n_pos_val < 50:
    print(f"\n⚠️  H1 CONFIRMÉE (partielle) : seulement {n_pos_val} positifs en val → PR-AUC peu fiable.")
else:
    print(f"\n✅ H1 rejetée : {n_pos_val} positifs en val, score statistiquement valide.")

In [ ]:
# H2 — Déviation des features MACH-07 vs flotte
# Pour chaque feature, calcule le z-score de la moyenne MACH-07 par rapport aux moyennes des autres machines

fleet_means  = trainval_df.groupby("machine_id")[FEATURE_COLS].mean()
fleet_std    = fleet_means.std()           # std inter-machines
fleet_global = fleet_means.mean()         # moyenne de la flotte

mach07_mean  = fleet_means.loc["MACH-07"]
others_mean  = fleet_means.drop("MACH-07").mean()

# Z-score : combien de std MACH-07 est éloignée des autres machines ?
z = ((mach07_mean - fleet_global) / fleet_std.replace(0, np.nan)).abs().dropna()
top_outlier = z.sort_values(ascending=False).head(15)

print("=== H2 — Top 15 features les plus déviantes pour MACH-07 ===")
print(f"  (|z| = nb d'écarts-types vs moyenne flotte)\n")
print(f"  {'Feature':<45} {'|z|':>6}  {'MACH-07':>10}  {'Flotte':>10}")
print("  " + "-"*75)
for feat, z_val in top_outlier.items():
    m7  = mach07_mean[feat]
    flt = fleet_global[feat]
    flag = " ⚠" if z_val > 2 else ""
    print(f"  {feat:<45} {z_val:>6.2f}  {m7:>10.3f}  {flt:>10.3f}{flag}")

n_extreme = (z > 2).sum()
print(f"\n  Features avec |z| > 2 : {n_extreme} / {len(z)}")
if n_extreme > 5:
    print("⚠️  H2 PLAUSIBLE : MACH-07 présente plusieurs features très atypiques.")
else:
    print("✅ H2 peu probable : distribution features proche de la flotte.")

In [ ]:
# H3 — Le modèle prédit-il dans le mauvais sens sur MACH-07 ?
gkf2 = GroupKFold(n_splits=N_MACHINES)
mach07_fold = None
for tr_idx, vl_idx in gkf2.split(X_tv, y_tv, groups):
    if groups.iloc[vl_idx].unique()[0] == "MACH-07":
        mach07_fold = (tr_idx, vl_idx)
        break

tr_idx, vl_idx = mach07_fold
X_tr07, y_tr07 = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
X_vl07, y_vl07 = X_tv.iloc[vl_idx], y_tv.iloc[vl_idx]

spw07 = round((y_tr07 == 0).sum() / max((y_tr07 == 1).sum(), 1), 2)
pipe07 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   XGBClassifier(**{**B8_PARAMS, "scale_pos_weight": spw07})),
])
pipe07.fit(X_tr07, y_tr07)
y_prob07 = pipe07.predict_proba(X_vl07)[:, 1]

pos_scores = y_prob07[y_vl07 == 1]
neg_scores = y_prob07[y_vl07 == 0]

print("=== H3 — Distribution des scores prédits sur MACH-07 ===")
print(f"\n  Positifs (pannes réelles)  n={len(pos_scores)}")
print(f"    Médiane score : {np.median(pos_scores):.3f}")
print(f"    Moyenne score : {np.mean(pos_scores):.3f}")
print(f"    % scores > 0.5 : {(pos_scores > 0.5).mean()*100:.1f}%")
print(f"\n  Négatifs (pas de panne)    n={len(neg_scores)}")
print(f"    Médiane score : {np.median(neg_scores):.3f}")
print(f"    Moyenne score : {np.mean(neg_scores):.3f}")
print(f"    % scores > 0.5 : {(neg_scores > 0.5).mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(neg_scores, bins=30, alpha=0.7, color="#9B7FD4", label="Négatifs (0)")
axes[0].hist(pos_scores, bins=30, alpha=0.8, color="#E05A5A", label="Positifs (pannes)")
axes[0].axvline(0.5, color="white", linestyle="--", linewidth=1)
axes[0].set_xlabel("Score prédit")
axes[0].set_ylabel("Fréquence")
axes[0].set_title("MACH-07 — distribution des scores")
axes[0].legend(fontsize=9)

bp = axes[1].boxplot([neg_scores, pos_scores],
                     patch_artist=True,
                     boxprops=dict(facecolor="#9B7FD4", alpha=0.5),
                     medianprops=dict(color="#E8B84B", linewidth=2))
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(["Négatifs", "Positifs (pannes)"])
axes[1].axhline(0.5, color="#E05A5A", linestyle="--", linewidth=1, label="Seuil 0.5")
axes[1].set_ylabel("Score prédit")
axes[1].set_title("MACH-07 — boxplot positifs vs négatifs")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("mach07_analysis.png", dpi=150)
plt.show()

sep = np.mean(pos_scores) - np.mean(neg_scores)
print(f"\n  Δ moyen (pos − neg) : {sep:+.3f}")
if sep < 0.05:
    print("⚠️  H3 CONFIRMÉE : le modèle n'arrive pas à séparer pannes et non-pannes pour MACH-07.")
    print("   → Les pannes MACH-07 ressemblent aux observations normales des autres machines.")
else:
    print("✅ H3 rejetée : le modèle fait bien la distinction, problème ailleurs.")